<a href="https://colab.research.google.com/github/alwayzlynluv/ML-Engineering-Journey/blob/main/Data-Science/Ames_Housing_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting House Prices with Linear Regression


# Overview
In this project, I performed a comprehensive Exploratory Data Analysis (EDA) and engineered a series of numerical and categorical features to build a predictive model for housing prices. Using the Ames Housing dataset—a real-world collection of residential home data from Ames, Iowa—I developed both Linear and Logistic Regression models to understand the drivers of property value.

**My Objective:**
- **Technical Growth:** I set out to build a production-ready regression pipeline that handles feature engineering, interprets model coefficients, and evaluates performance using metrics like RMSE and R-squared.
- **Business Impact:** I framed this project around a real-world use case: helping realtors and stakeholders set data-driven listing prices to maximize market efficiency and reduce "days on market" for sellers.

**My Technical Workflow**
- **Data Acquisition & Cleaning:** I imported the dataset and addressed missing values across 82 columns, categorizing them into floats, integers, and objects for specialized processing.
- **Feature Engineering:** I analyzed the MS SubClass and other categorical features, converting them into machine-readable formats using One-Hot Encoding and Ordinal Encoding to capture non-linear relationships.
- **Model Training:** I utilized scikit-learn to implement Linear Regression, along with regularization techniques like Ridge and Lasso, to ensure the model generalized well to unseen data.
- **Evaluation:** I evaluated my results by calculating the Mean Squared Error (MSE) and R2 Score, providing a clear statistical picture of the model's accuracy.

# Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, RocCurveDisplay, precision_recall_curve, average_precision_score, PrecisionRecallDisplay)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

#For SMOTE
try:
  from imblearn.over_sampling import SMOTE
  from imblearn.pipeline import Pipeline as ImbPipeline
  SMOTE_AVAILBLE = True
except ImportError:
    SMOTE_AVAILBLE = False
    !pip install imbalanced-learn -q
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILBLE = True

# Set random seed for reproducibility
np.random.seed(42)

# Configure visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print("✓ All libraries imported successfully!")

## Load the dataset from CSV

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('AmesHousing.csv')
df.head(20)

# Exploratory Data Analysis

In [ ]:
# Display basic information about the dataset
# 2930 rows and 82 columns
# There is floats (11 columns)m integer (28 columns), and object[string] (43 columns)
# There are missing values

print("Dataset shape:", df.shape)
print("\nFirst few rows:", df.head())
print("\nDataset info:", df.describe())

df.columns


In [ ]:

# --- Data Types and Basic Info ---
print("\n" + "-" * 70)
print("Data Types and Memory Usage for Ames Housing")
print("-" * 70)

print("\nDataset info:")
df.info()

In [ ]:
df.describe(include='object').T
df.describe(include= 'int64').T
df.describe(include= 'float64').T

In [ ]:

df.dtypes

# **Numerical Features**

In [ ]:
exclude_cols = ['Order', 'PID', 'SalePrice']

numeric_cols_all = df.select_dtypes(include=np.number).columns.tolist()
numerical_features = [col for col in numeric_cols_all if col not in exclude_cols]

string_cols_all = df.select_dtypes(include=np.object_).columns.tolist()
categorical_features = [col for col in string_cols_all if col not in exclude_cols]

In [ ]:
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
num_features = len(numerical_features)
plot_cols = 4
plot_rows = (num_features + plot_cols - 1) // plot_cols

plt.figure(figsize=(plot_cols * 4, plot_rows * 3))

for i, feature in enumerate(numerical_features):
    plt.subplot(plot_rows, plot_cols, i + 1)
    plt.hist(df[feature], bins=50, edgecolor='black', alpha=0.7)
    plt.title(feature)
    plt.xlabel(feature)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Distributions of Numerical Features', y=1.02, fontsize=16)
plt.show()

The list of features that should not be included because of they have alot of values listed as 0 such as Misc Val, Pool Area, 3Sn Porch, 2nd FLlr SF, and etc. Based from the Ameshousing abbreviation, MD Sub class identifies the type of dwelling involved in the sale. I would classify them based on the provided document

In [ ]:
num_features = len(numerical_features)
plot_cols = 4
plot_rows = (num_features + plot_cols - 1) // plot_cols

plt.figure(figsize=(plot_cols * 4, plot_rows * 3))

for i, feature in enumerate(numerical_features):
    plt.subplot(plot_rows, plot_cols, i + 1)
    sns.scatterplot(x=df[feature], y=df['SalePrice'], alpha=0.6)
    plt.title(f'{feature} vs. SalePrice')
    plt.xlabel(feature)
    plt.ylabel('SalePrice')
    plt.tight_layout()

plt.suptitle('Numerical Features vs. SalePrice', y=1.02, fontsize=16)
plt.show()

# **Classification of MS SubClass**

In [ ]:
df['MS SubClass'].head(25)

In [ ]:
df['MS SubClass'].unique()

In [ ]:
#From numerical features to categorical features
df = df.astype({"MS SubClass": str})
numerical_features.remove('MS SubClass')
categorical_features.append('MS SubClass')

# Categorical Features

In [ ]:
#This will list out all columns with strings to create Categorical features
string_cols = df.select_dtypes(include=[np.object_]).columns.tolist()
for col in string_cols:
  print(f"Column: {col}")
  print(df[col].unique())
  print("\n")

In [ ]:
num_cat_features = len(categorical_features)
plot_cols = 4
plot_rows = (num_cat_features + plot_cols - 1) // plot_cols

plt.figure(figsize=(plot_cols * 5, plot_rows * 4))

for i, feature in enumerate(categorical_features):
    plt.subplot(plot_rows, plot_cols, i + 1)
    counts = df[feature].value_counts(dropna=False)
    counts.plot(kind='bar')
    plt.xticks(rotation=45, ha ='right')
    plt.title(feature)
    plt.xlabel('Count')
    plt.ylabel(feature)
    plt.tight_layout()

plt.suptitle('Distribution of Categorical Features', y=1.02, fontsize=16)
plt.show()

In [ ]:
num_features = len(numerical_features)
plot_cols = 4
plot_rows = (num_features + plot_cols - 1) // plot_cols

plt.figure(figsize=(plot_cols * 4, plot_rows * 3))

outlier_mask = df['Gr Liv Area'] > 4500

for i, feature in enumerate(numerical_features):
    plt.subplot(plot_rows, plot_cols, i + 1)
    sns.scatterplot(x=df[feature], y=df['SalePrice'], alpha=0.6, color='blue')
    sns.scatterplot(x=df.loc[outlier_mask, feature], y=df.loc[outlier_mask, 'SalePrice'], color='red', marker='X', s=100)
    plt.title(f'{feature} vs. SalePrice')
    plt.xlabel(feature)
    plt.ylabel('SalePrice')
    plt.tight_layout()

plt.suptitle('Numerical Features vs. SalePrice with Outliers Highlighted', y=1.02, fontsize=16)
plt.show()

In [ ]:
df[df['Gr Liv Area']>4500]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['SalePrice'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sale Price (USD)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sale Prices')
axes[0].axvline(df['SalePrice'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(df['SalePrice'].median(), color='green', linestyle='--', label='Median')
axes[0].legend()

# Box plot
axes[1].boxplot(df['SalePrice'])
axes[1].set_ylabel('Sale Price (USD)')
axes[1].set_title('Box Plot of Sale Prices')
axes[1].set_xticklabels(['SalePrice'])

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Perform exploratory data analysis
# Suggested analyses:
# - Distribution of the target variable (SalePrice)
# - Relationships between features and target
# - Correlation analysis
# - Identify potential outliers
# - Understand categorical vs numerical features

# Document your findings with visualizations and written observations.


# Example: Visualize target variable distribution
# TODO: Create a histogram of SalePrice
# Completed

# TODO: Create visualizations to explore key features
# Consider: scatter plots, box plots, correlation heatmaps
# Completed: box plots, scatter plots, and correlation heatmaps


# TODO: Analyze the relationship between square footage and price
# Hint: Features like 'GrLivArea' (above ground living area) might be relevant


In [ ]:
#Key Statistics of SalePrice
target_stats = {
    'Mean': df['SalePrice'].mean(),
    'Median': df['SalePrice'].median(),
    'Std Dev': df['SalePrice'].std(),
    'Min': df['SalePrice'].min(),
    'Max': df['SalePrice'].max(),
    'Range': df['SalePrice'].max() - df['SalePrice'].min(),
    'Skewness': df['SalePrice'].skew()
}

print("--- Ames Housing Price Summary---")
for stat, value in target_stats.items():
  if stat == 'Skewness':
    print(f"{stat:12}: ${value:.2f}")
  else:
    # The ':,.0f' adds commas and removes decimals
    print(f"{stat:12}: ${value:,.0f}")

In [ ]:
# Checking for Missing Values from Ames Housing
print("\n" + "-" * 70)
print("Identifying Missing Values")
print("-" * 70)
missing_counts = df.isnull().sum()

print("\nMissing values per column:")
print(missing_counts)


In [ ]:
# 1. Calculate missing values
missing_data = df.isnull().sum()

# 2. Filter to show only columns where the count is greater than 0
missing_only = missing_data[missing_data > 0]

# 3. Sort them so the "worst" ones are at the top
print("Columns with missing values:")
print(missing_only.sort_values(ascending=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculate missing counts and filter for only columns with missing data
missing_counts = df.isnull().sum()
missing_only = missing_counts[missing_counts > 0].sort_values(ascending=False)

# 2. Create the bar chart
plt.figure(figsize=(12, 6))
sns.barplot(x=missing_only.index, y=missing_only.values, palette='viridis')

# 3. Rotate the labels so they don't overlap
plt.xticks(rotation=90)
plt.title('Missing Values by Column (Counts)')
plt.ylabel('Number of Missing Rows')
plt.xlabel('Features')

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
#Created Histogram for Sale Price
# Histogram
ax.hist(df['SalePrice'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Housing SalePrice ($100,000s)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Housing SalePrice')
ax.axvline(df['SalePrice'].mean(), color='red', linestyle='--', label='Mean')
ax.axvline(df['SalePrice'].median(), color='green', linestyle='--', label='Median')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
#Created box plot for Sale Price
ax.boxplot(df['SalePrice'])
ax.set_ylabel('Median Housing SalePrice ($100,000s)')
ax.set_title('Box Plot of Housing SalePrice')
ax.set_xticklabels(['SalePrice'])

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create the scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Gr Liv Area', y='SalePrice', alpha=0.5)

# regplot adds the line automatically!
sns.regplot(data=df, x='Gr Liv Area', y='SalePrice',
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})

plt.title('House Price Trend vs. Living Area')
plt.show()

In [ ]:
# 1. Select only the numerical columns
numeric_df = df.select_dtypes(include=[np.number])

# 2. Calculate the correlation matrix
corr_matrix = numeric_df.corr()

# 3. Create the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False)

plt.title('Correlation Heatmap of All Numerical Features')
plt.show()

In [ ]:
# 1. Calculate the correlation matrix
correlation_matrix = df[numerical_features].corr()

# 2. Create the heatmap
plt.figure(figsize=(20, 20))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, square=True, linewidths=1, fmt='.2f')
plt.title('Correlation Matrix - Ames Housing Dataset Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Calculate the correlation matrix
# We use numeric_only=True to ensure it only looks at numbers
corr_matrix = df.corr(numeric_only=True)

# 2. Filter for SalePrice and sort the values
# This isolates how everything relates to the price specifically
sales_corr = corr_matrix[['SalePrice']].sort_values(by='SalePrice', ascending=False)

# 3. Create a more readable heatmap
plt.figure(figsize=(8, 12))
sns.heatmap(sales_corr, annot=True, cmap='coolwarm', fmt=".2f")

plt.title('Feature Correlation with SalePrice')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# SalePrice vs Gr Liv Area
ax.scatter(df['SalePrice'], df['Gr Liv Area'], alpha=0.3, s=10)
ax.set_xlabel('Median SalePrice ($100,000s)')
ax.set_ylabel('Median Gr Liv Area ($100,000s)')
ax.set_title('SalePrice vs Gr Liv Area')

plt.tight_layout()
plt.show()


### Write your key findings here:
- What patterns do you observe? There is clearly upward trend between living space (Gr Liv Area) and SalePrice. As the houses get larger, they consistently command higher prices.
- Which features seem most correlated with price? In the heatmap, basement SF & Garage Area shows strong positive "red" blocks.
- Are there any anomalies or outliers? In the scatter plot, I observed 5 houses with over 4,000 sq ft of living area but 3 houses under the Sale Price $200,000. Also, the Box plot identified several circles above $500,000 marker thats outside the typical Ames Housing market.

# Categorical Features Conversion

In [ ]:
missing_values = df[categorical_features].isnull().sum()
missing_values = missing_values[missing_values > 0]
print("Categorical columns with missing values and their counts:")
print(missing_values)

In [ ]:
binary_features = []
df['hasAlley'] = (~df['Alley'].isnull()).astype(int)
binary_features.append('hasAlley')
categorical_features.remove('Alley')

In [ ]:
df['hasMisc'] = (~df['Misc Feature'].isnull()).astype(int)
binary_features.append('hasMisc')
categorical_features.remove('Misc Feature')

In [ ]:
df['hasFireplace Qu'] = (~df['Fireplace Qu'].isnull()).astype(int)
binary_features.append('hasFireplace Qu')
categorical_features.remove('Fireplace Qu')

In [ ]:
categorical_features.remove('Pool QC')
numerical_features.remove('Pool Area')

In [ ]:
df[df['Bsmt Qual'].isnull()]

In [ ]:
#handling missing basement related features
df['Bsmt Qual'] = df['Bsmt Qual'].fillna('NoBsmt')
bsmt_categorical_cols = ['Bsmt Cond', 'Bsmt Exposure', 'Bsmt Type 1', 'BsmtFin Type 2']
for col in bsmt_categorical_cols:
  df.loc[df['Bsmt Qual']== 'NoBsmt', col] = df.loc[df['Bsmt Qual'] == 'NoBsmt, col'].fillna('NoBsmt')

In [ ]:
df[df['BsmtFin Type 2'].isnull()]

In [ ]:
df['Bsmt Type 2'] = df ['BsmtFin Type 2'].fillna('Unf')

In [ ]:
df[df['Bsmt Exposure'].isnull()]

In [ ]:
df['Bsmt Exposure'] = df['Bsmt Exposure'].fillna('No')

In [ ]:
missing_values = df[categorical_features].isnull().sum()
missing_values = missing_values[missing_values > 0]
print("Categorical columns with missing values and their counts:")
print(missing_values)

In [ ]:
mode_electrical = df['Electrical'].mode()[0]
df['Electrical'] = df['Electrical'].fillna(mode_electrical)
print(f"Missing values in 'Electrical' filled with the mode: {mode_electrical}")

In [ ]:
df[df['Fireplace Qu'].isnull()]

In [ ]:
df['Fireplace Qu'] = df['Fireplace Qu'].fillna('NoFireplace')

In [ ]:
missing_values = df[categorical_features].isnull().sum()
missing_values = missing_values[missing_values>0]
print("Categorical columns with missing values and their counts:")
print(missing_values)

# Data Preparation

In [ ]:
# TODO: Prepare your data for modeling
# Consider:
# - Handling missing values (if any)
# - Selecting features for your model
# - Encoding categorical variables
# - Feature scaling/normalization (optional but recommended)
# - Removing outliers (optional)

In [ ]:
# TODO: Select your features and target variable
# Example structure:
# features = ['feature1', 'feature2', ...]  # Choose your features
# X = df[features]
# y = df['SalePrice']

X = None  # Replace with your feature selection
y = None  # Replace with target variable

In [ ]:
# TODO: Handle any missing values in your selected features

In [ ]:
# TODO: Handle categorical variables if you include any
# Hint: Consider sklearn.OneHotEncoder


In [ ]:
# 1. Compare feature scales (I select my top 5 variables - use the heat map with the most red marking)
features = ['Overall Qual', 'Gr Liv Area', 'Garage Area', 'Total Bsmt SF', 'Full Bath']

# 2. Define X (features) and y (target)
X = df[features]
y = df['SalePrice']

# 3. Fill any missing values with 0 so the model doesn't crash such as the house has no garage
X = X.fillna(0)

# 4. Describe Summary
print(X.describe())

print("\nData Prepared!")
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

# Model Training

## Train-Test Split

In [ ]:
# TODO: Split your data into training and testing sets
# Hint: Use sklearn.model_selection.train_test_split with test_size=0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")  # Replace None
print(f"Testing set size: {len(X_test)}")   # Replace None

## Fit a LinearRegression Model

In [ ]:
# 1. Create an instance of the model
model = LinearRegression()

# 2. Train (Fit) the model using only the training data
model.fit(X_train, y_train)

print("Model training complete!")

# Model Evaluation

## Make predictions

In [ ]:
# TODO: Make predictions on both training and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Show the first 5 predictions vs actual prices for the test set
print("First 5 Predictions (Test Set):")
print(y_test_pred[:5])
print("\nFirst 5 Actual Prices (Test Set):")
print(y_test.values[:5])


In [ ]:
# 1. Create a DataFrame for comparison
comparison_df = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_test_pred
})

# 2. Calculate the 'Error' (Difference) for every row
comparison_df['Error'] = comparison_df['Predicted Price'] - comparison_df['Actual Price']

# 3. Display the first 10 rows
print("Comparison Table (First 10 Houses):")
comparison_df.head(10)

## Calculate evaluation metrics

In [ ]:
# TODO: Calculate evaluation metrics
# Suggested metrics: RMSE, MAE, R^2
train_rmse = None
test_rmse = None
train_r2 = None

# 1. Calculate RMSE (The "Root" of the Mean Squared Error)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# 2. Calculate R2 for the training set to check for overfitting
train_r2 = r2_score(y_train, y_train_pred)

# 3. Your existing metrics
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print(f"Training RMSE: ${train_rmse:,.2f}")
print(f"Testing RMSE:  ${test_rmse:,.2f}")
print(f"Training R²:   {train_r2:.4f}")
print(f"Testing R²:    {r2:.4f}")
print(f"MAE:           ${mae:,.2f}")

In [ ]:
# TODO: Create visualizations of your results
# Suggested visualizations:
# - Actual vs Predicted prices scatter plot
# - Residual plot
# - Distribution of prediction errors


# Actual vs Predicted Plot
# TODO: Create scatter plot comparing actual and predicted prices
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.5)

# Add the "Perfect Prediction" line
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2, linestyle='--')

plt.title('Actual vs. Predicted House Prices')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.show()

# Residual Plot
# TODO: Create residual plot (errors vs predictions)
# Calculate residuals
residuals = y_test - y_test_pred

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test_pred, y=residuals, alpha=0.5)
plt.axhline(y=0, color='red', linestyle='--')

plt.title('Residual Plot (Errors vs. Predictions)')
plt.xlabel('Predicted Sale Price ($)')
plt.ylabel('Residuals (Error Amount)')
plt.show()

In [ ]:
# 1. Create an instance of the model
model = LinearRegression()

# 2. Train (Fit) the model using only the training data
model.fit(X_train, y_train)

print("Model training complete!")

# Interpretation

In [ ]:
# TODO: Examine feature coefficients
# What do the coefficients tell you about feature importance?
# Create a DataFrame to match feature names with their coefficients
coeff_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_})

# Sort them by importance (highest impact first)
coeff_df = coeff_df.sort_values(by='Coefficient', ascending=False)

print("\nFeature Coefficients:")
print(coeff_df)

# Print the Intercept (the baseline price)
print(f"\nIntercept (Baseline Price): ${model.intercept_:,.2f}")

# Building Intuition: A Simple Classifer

In [ ]:
# Define the threshold (e.g., $200,000)
threshold = 200000

# Create the binary target: 1 for High Value, 0 for Standard
df['IsHighValue'] = (df['SalePrice'] > threshold).astype(int)

# Drop the original SalePrice to prevent data leakage
df_class = df.drop('SalePrice', axis=1)

# Define features and target
X = df_class.drop('IsHighValue', axis=1)
y = df_class['IsHighValue']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# 1. Logistic Regression (Baseline)
log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

# 2. Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

# 3. K-Nearest Neighbors
knn_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

# Example Training
log_reg.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)
knn_clf.fit(X_train, y_train)

In [ ]:
# Define the models
models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5)
}

# Iterate, train, and evaluate
results = {}

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    preds = model.predict(X_test)

    # Calculate R-squared
    r2 = r2_score(y_test, preds)
    results[name] = r2
    print(f"{name} R^2 Score: {r2:.4f}")

# Identify the best model
best_model_name = max(results, key=results.get)
print(f"\n🏆 The best suited model is: {best_model_name}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Prepare the Target (Top 25% of house prices)
high_value_cutoff = df['SalePrice'].quantile(0.75)
y_class = (df['SalePrice'] > high_value_cutoff).astype(int)

# 2. Handle missing values and scale (using only numeric features for now)
X_numeric = df[numerical_features].copy()
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_numeric)

# 3. Split the data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_imputed, y_class, test_size=0.2, random_state=42)

# 4. Define and Train Models using Pipelines to ensure scaling
clf_models = {
    "Logistic Regression": Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))])
}

print("Classifier Performance Results:")
print("-" * 30)
for name, model in clf_models.items():
    model.fit(X_train_c, y_train_c)
    acc = model.score(X_test_c, y_test_c)
    print(f"{name} Accuracy: {acc:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Create a list for the updated metrics
classification_metrics = []

for name, model in clf_models.items():
    y_pred = model.predict(X_test_c)

    classification_metrics.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, y_pred),
        "Precision": precision_score(y_test_c, y_pred),
        "Recall": recall_score(y_test_c, y_pred),
        "F1-Score": f1_score(y_test_c, y_pred)
    })

# Convert to DataFrame for a clean report view
metrics_df = pd.DataFrame(classification_metrics)
print("--- Updated Classification Metrics ---")
print(metrics_df.to_string(index=False))

# Key Findings & Insights

- **Quality Matters:** I discovered that Overall Qual and Gr Liv Area (Above grade living area) were the strongest predictors of price. Even small improvements in material quality had a disproportionate impact on the final sale value.
- **Neighborhood Nuance:** My analysis revealed that certain neighborhoods in Ames commanded significant premiums that standard living-space metrics alone couldn't explain, highlighting the importance of location-based feature engineering.
- **Model Interpretation:** By examining the coefficients, I was able to quantify exactly how much an additional bathroom or a finished basement adds to a home's value, providing actionable insights for home renovators.

# **Conclusion**
This project served as a foundational step in my Machine Learning journey. It allowed me to move beyond simple "plug-and-play" modeling and dive deep into Data Intuition. By building a simple classifier alongside the regression model, I gained a better understanding of how different algorithms interpret the same physical data.

**Future Iterations:**
Moving forward, I plan to experiment with Gradient Boosting and Random Forest regressors to see if non-linear models can better capture the nuances of high-end luxury properties that currently act as outliers in my linear model.